# Results analysis

Reads the CSV files written by the scripts in `scripts/` and looks a bit deeper than the README:
per-object breakdown, failure types, what the world model gets wrong and where the planner spends its time.
Nothing here re-runs simulations.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rwm.evaluation import plots
from rwm.evaluation.benchmark import summarise

plots.setup()
R = Path("../results")
pd.set_option("display.precision", 3)

## 1. Benchmark with ground-truth object state

In [2]:
gt = pd.read_csv(R / "benchmark/ground_truth/episodes.csv")
summary = summarise(gt)
summary[["condition", "controller", "episodes", "success_rate", "success_ci_low", "success_ci_high",
         "place_error_success_mm", "collision_steps", "time_to_success_steps", "act_time_ms"]]

,condition,controller,episodes,success_rate,success_ci_low,success_ci_high,place_error_success_mm,collision_steps,time_to_success_steps,act_time_ms
0,nominal,scripted,50,1.000,0.929,1.000,1.883,0.000,59.520,0.030
1,new_positions,scripted,50,1.000,0.929,1.000,2.936,0.000,67.280,0.033
2,new_objects,scripted,50,1.000,0.929,1.000,1.149,0.000,57.720,0.035
3,heavy_slippery,scripted,50,1.000,0.929,1.000,2.363,0.000,58.160,0.030
4,nominal,mpc,50,0.860,0.738,0.930,5.201,0.020,40.372,193.816
5,nominal,mpc_unc,50,0.800,0.670,0.888,5.444,0.580,39.125,187.797
6,new_positions,mpc,50,0.640,0.501,0.759,10.035,0.940,55.594,187.183
7,new_positions,mpc_unc,50,0.580,0.442,0.706,10.851,2.540,63.517,192.244
8,new_objects,mpc,50,0.700,0.562,0.809,5.883,1.220,40.200,186.147
9,new_objects,mpc_unc,50,0.880,0.762,0.944,8.033,0.020,37.068,191.044


SAC is evaluated with three training seeds, each on the same 50 scenarios per condition. The spread between
seeds is as large as the binomial interval of a single seed:

In [3]:
sac = gt[gt.controller == "sac"]
sac.groupby(["condition", "seed"]).success.mean().unstack()

seed,0,1,2
condition,,,
heavy_slippery,0.90,0.84,0.74
new_objects,0.58,0.68,0.80
new_positions,0.08,0.28,0.12
nominal,0.94,0.92,0.72


### Per object type

In [4]:
gt.groupby(["condition", "object_kind", "controller"]).success.mean().unstack().round(2)

controller                   mpc  mpc_unc   sac  scripted
condition      object_kind                               
heavy_slippery box          0.83     0.93  0.83       1.0
               cylinder     0.86     1.00  0.83       1.0
new_objects    box          0.48     0.76  0.70       1.0
               sphere       0.86     0.97  0.68       1.0
new_positions  box          0.44     0.60  0.13       1.0
               cylinder     0.84     0.56  0.19       1.0
nominal        box          0.78     0.70  0.88       1.0
               cylinder     0.93     0.89  0.84       1.0

### How do failures look?

`dropped` means the object left the table, `max_lift < 1 cm` means the object was never really lifted
(missed or slipping grasp), otherwise the object was lifted but not left on the platform at the end.

In [5]:
def failure_type(r):
    if r.success:
        return "success"
    if r.dropped:
        return "object dropped off the table"
    if r.max_lift < 0.01:
        return "never lifted"
    if r.ever_succeeded:
        return "placed, then disturbed"
    return "lifted, not placed"

gt["outcome"] = gt.apply(failure_type, axis=1)
gt.groupby(["controller", "outcome"]).size().unstack(fill_value=0)

outcome,"lifted, not placed",never lifted,object dropped off the table,"placed, then disturbed",success
controller,,,,,
mpc,37,8,0,3,152
mpc_unc,31,3,2,3,161
sac,45,172,0,3,380
scripted,0,0,0,0,200


## 2. Camera in the loop

In [6]:
cam = pd.read_csv(R / "benchmark/camera_realistic/episodes.csv")
cam["outcome"] = cam.apply(failure_type, axis=1)
both = pd.concat([gt[gt.condition == "nominal"].assign(observation="ground truth"), cam.assign(observation="camera")])
both.groupby(["controller", "observation"]).success.mean().unstack().round(2)

observation,camera,ground truth
controller,,
mpc,0.68,0.86
mpc_unc,0.64,0.80
sac,0.60,0.86
scripted,0.96,1.00


Where the camera hurts (nominal condition, ground truth vs camera). The two learned controllers do not fail at
the same place: SAC's extra failures happen after the grasp (the object is lifted but does not end up on the
platform), MPC's at the grasp itself (the object is never lifted). While the object is in the hand its pose is
extrapolated from the gripper and its velocity from finite differences, two quantities SAC never saw with that
kind of error; before the grasp, MPC needs the object pose to be consistent with what its model was trained on.

In [7]:
nominal = pd.concat([gt[gt.condition == "nominal"].assign(observation="ground truth"), cam.assign(observation="camera")])
nominal["outcome"] = nominal.apply(failure_type, axis=1)
nominal.groupby(["controller", "observation", "outcome"]).size().unstack(fill_value=0)

outcome                  lifted, not placed  never lifted  \
controller observation                                      
mpc        camera                         6             9   
           ground truth                   6             1   
mpc_unc    camera                        11             5   
           ground truth                   8             1   
sac        camera                        38            19   
           ground truth                   6            15   
scripted   camera                         0             2   
           ground truth                   0             0   

outcome                  object dropped off the table  placed, then disturbed  \
controller observation                                                          
mpc        camera                                   0                       1   
           ground truth                             0                       0   
mpc_unc    camera                                   1                       1   
           ground truth                             1                       0   
sac        camera                                   1                       2   
           ground truth                             0                       0   
scripted   camera                                   0                       0   
           ground truth                             0                       0   

outcome                  success  
controller observation            
mpc        camera             34  
           ground truth       43  
mpc_unc    camera             32  
           ground truth       40  
sac        camera             90  
           ground truth      129  
scripted   camera             48  
           ground truth       50

## 3. World model

In [8]:
err = pd.read_csv(R / "world_model/multistep_errors.csv")
err = err[err.contact.notna()]
err.pivot_table(index=["feature", "contact"], columns="horizon", values=["model_median", "constant_median"]).round(2)

constant_median                 model_median         \
horizon                              1       5       10           1      5    
feature         contact                                                       
arm_q_deg       False              0.57    2.09    3.59         0.05   0.19   
                True               0.93    3.28    5.35         0.06   0.24   
grip_mm         False              3.28    4.58    5.09         0.33   0.38   
                True               0.14    0.46    3.97         0.60   1.57   
obj_linvel_mm_s False              0.04    0.04    0.05         4.90  10.07   
                True              73.05  192.29  250.34        14.60  25.07   
obj_pos_mm      False              0.00    0.00    0.00         0.28   1.60   
                True              10.57   42.15   94.29         0.75   3.62   
obj_rot_deg     False              0.00    0.00    0.00         0.25   1.29   
                True               0.46    1.67    2.79         0.43   1.96   
tcp_pos_mm      False              7.78   28.47   50.16         0.62   2.53   
                True              18.36   61.25   99.10         0.81   3.22   

                                
horizon                     10  
feature         contact         
arm_q_deg       False     0.35  
                True      0.40  
grip_mm         False     0.41  
                True      2.43  
obj_linvel_mm_s False    10.59  
                True     30.78  
obj_pos_mm      False     3.31  
                True      7.71  
obj_rot_deg     False     2.50  
                True      3.75  
tcp_pos_mm      False     4.69  
                True      5.54

The error grows roughly linearly with the horizon in free motion but much faster when the fingers touch the
object: the grasp is where the dynamics are discontinuous (the object starts following the gripper only once
the fingers squeeze it, and slips if the friction is low).

In [9]:
calib = pd.read_csv(R / "world_model/uncertainty_vs_error.csv")
calib["spread_bin"] = pd.qcut(calib.spread_mm, 8)
calib.groupby("spread_bin", observed=True).error_mm.describe()[["count", "50%", "75%", "max"]].round(2)

,count,50%,75%,max
spread_bin,,,,
"(0.569, 1.011]",710.0,1.25,1.59,13.27
"(1.011, 1.19]",709.0,1.19,1.77,22.47
"(1.19, 1.376]",709.0,1.42,2.11,29.36
"(1.376, 1.652]",709.0,1.79,2.41,53.21
"(1.652, 2.087]",709.0,2.26,3.63,47.74
"(2.087, 2.794]",709.0,2.84,4.91,57.94
"(2.794, 4.507]",709.0,4.96,8.88,121.03
"(4.507, 46.372]",709.0,13.76,24.46,152.96


In [10]:
hist = pd.read_csv(R / "world_model/training_history.csv")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(hist.epoch, hist.train_mse, color=plots.COLORS["scripted"], label="train")
ax.plot(hist.epoch, hist.val_mse, color=plots.COLORS["sac"], label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE on normalised state change"); ax.legend()
ax.set_title("Ensemble training (early stopping on validation)", loc="left");

## 4. Uncertainty penalty

In [11]:
sweep_file = R / "benchmark/uncertainty_sweep/summary.csv"
if sweep_file.exists():
    sweep = pd.read_csv(sweep_file)
    display(sweep[["condition", "uncertainty_weight", "episodes", "success_rate", "success_ci_low",
                   "success_ci_high", "collision_steps", "time_to_success_steps"]])

,condition,uncertainty_weight,episodes,success_rate,success_ci_low,success_ci_high,collision_steps,time_to_success_steps
0,new_objects,0.00,50,0.70,0.562,0.809,1.22,40.200
1,new_objects,0.02,50,0.80,0.670,0.888,0.92,34.875
2,new_objects,0.05,50,0.88,0.762,0.944,0.02,37.068
3,new_objects,0.10,50,0.90,0.786,0.957,0.02,41.511
4,new_objects,0.20,50,0.68,0.542,0.792,0.12,43.647
5,nominal,0.00,50,0.86,0.738,0.930,0.02,40.372
6,nominal,0.02,50,0.88,0.762,0.944,0.00,38.409
7,nominal,0.05,50,0.80,0.670,0.888,0.58,39.125
8,nominal,0.10,50,0.78,0.648,0.872,0.10,42.077
9,nominal,0.20,50,0.66,0.522,0.776,0.40,56.667


## 5. Compute

In [12]:
lat = gt[gt.controller.str.startswith("mpc")].groupby("controller").plan_time_ms.describe()[["mean", "50%", "max"]]
lat

,mean,50%,max
controller,,,
mpc,190.471,189.178,276.854
mpc_unc,173.706,187.839,200.855


In [13]:
rl = sorted((R / "rl").glob("*_validation.csv"))
pd.DataFrame([{"run": f.stem.replace("_validation", ""), "steps": pd.read_csv(f).timestep.max(),
               "wall_time_h": pd.read_csv(f).wall_time.max() / 3600,
               "best_val_success": pd.read_csv(f).success_rate.max()} for f in rl])

,run,steps,wall_time_h,best_val_success
0,sac_demo_s0,300000,2.275,1.000
1,sac_demo_s1,300000,2.259,0.875
2,sac_demo_s2,300000,2.448,0.850
3,sac_s0,300000,2.611,0.000
